# 12 - Feature Removal Trade-off

This notebook measures the privacy-utility trade-off by progressively removing features that are most important for linkability.


## Same Data Constraint

This experiment uses the same segment-level rows for both tasks.

Base dataset:
- `utility_top150_dataset.csv.gz`

Procedure:
1. compute linkability importance on the same top-150 dataset;
2. rank original base features by linkability relevance;
3. remove the top-k linkability-sensitive features;
4. evaluate utility and linkability after each removal step.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from config import FEATURE_SETS_DIR
from modeling import compute_linkability_feature_importance, run_linkability_baselines, run_utility_baselines


In [2]:
REMOVAL_STEPS = [0, 10, 20, 30, 40, 50]
UTILITY_MODELS = ["LogisticRegression"]
LINKABILITY_SUMMARY_MODEL = "XGBoost"


## Load Utility Top-150 Dataset


In [3]:
top150_path = FEATURE_SETS_DIR / "utility_top150_dataset.csv.gz"
selected_features_path = FEATURE_SETS_DIR / "utility_top150_features.csv"

top150_features_df = pd.read_csv(top150_path, compression="gzip", low_memory=False)
selected_features_df = pd.read_csv(selected_features_path)
base_feature_columns = selected_features_df["feature"].tolist()

print("Top-150 dataset:", top150_features_df.shape)
print("Base feature count:", len(base_feature_columns))


Top-150 dataset: (406359, 157)
Base feature count: 150


## Rank Features By Linkability Importance


In [4]:
linkability_importance = compute_linkability_feature_importance(
    features_df=top150_features_df,
    model_name="LogisticRegression",
    test_size=0.2,
    random_state=42,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=4,
    permutation_scoring="roc_auc",
)

linkability_importance["model_based_df"].head(20)


,feature,importance
0,absdiff__lead_I_max,2.026364
1,absdiff__global_min_energy,1.957185
2,absdiff__global_mean_amplitude,1.831323
3,absdiff__lead_V4_min,1.614759
4,absdiff__lead_V3_skewness,1.591966
5,absdiff__lead_aVF_min,1.515571
6,absdiff__lead_II_max,1.371121
7,absdiff__lead_I_rms,1.353707
8,absdiff__global_min_min,1.322453
9,absdiff__lead_V1_min,1.258980


In [5]:
def pair_feature_to_base_feature(feature_name: str) -> str | None:
    if feature_name.startswith("absdiff__"):
        return feature_name.replace("absdiff__", "", 1)
    if feature_name.startswith("mean__"):
        return feature_name.replace("mean__", "", 1)
    return None

ranked_linkability_features = []
seen = set()
for feature_name in linkability_importance["model_based_df"]["feature"]:
    base_feature = pair_feature_to_base_feature(feature_name)
    if base_feature is None:
        continue
    if base_feature not in base_feature_columns:
        continue
    if base_feature in seen:
        continue
    seen.add(base_feature)
    ranked_linkability_features.append(base_feature)

ranked_linkability_features[:20]


['lead_I_max',
 'global_min_energy',
 'global_mean_amplitude',
 'lead_V4_min',
 'lead_V3_skewness',
 'lead_aVF_min',
 'lead_II_max',
 'lead_I_rms',
 'global_min_min',
 'lead_V1_min',
 'lead_V2_max',
 'lead_V2_min',
 'lead_III_kurtosis',
 'global_std_max',
 'global_std_zero_crossing_rate',
 'global_min_rms',
 'lead_aVR_abs_mean',
 'lead_aVL_energy',
 'lead_I_abs_mean',
 'lead_V1_abs_mean']

## Progressive Removal Experiment


In [6]:
removal_results = []

for n_remove in REMOVAL_STEPS:
    removed_features = ranked_linkability_features[:n_remove]
    kept_features = [feature for feature in base_feature_columns if feature not in removed_features]

    reduced_df = top150_features_df[
        ["patient_id", "segment_id", "label", "utility_label", "segment_ref", "start_sample", "end_sample"]
        + kept_features
    ].copy()

    utility_results = run_utility_baselines(
        features_df=reduced_df,
        target_col="utility_label",
        group_col="patient_id",
        test_size=0.2,
        random_state=42,
        models=UTILITY_MODELS,
    )
    linkability_results = run_linkability_baselines(
        features_df=reduced_df,
        group_col="patient_id",
        test_size=0.2,
        random_state=42,
        max_pairs=2000,
        representation="absdiff",
        min_segment_gap=4,
    )

    utility_row = utility_results["summary_df"].iloc[0]
    linkability_row = linkability_results["summary_df"]
    linkability_row = linkability_row[linkability_row["model"] == LINKABILITY_SUMMARY_MODEL].iloc[0]

    removal_results.append({
        "n_removed": n_remove,
        "n_kept": len(kept_features),
        "utility_model": utility_row["model"],
        "utility_f1": utility_row["f1_score"],
        "utility_balanced_accuracy": utility_row["balanced_accuracy"],
        "utility_roc_auc": utility_row["roc_auc"],
        "utility_pr_auc": utility_row["pr_auc"],
        "linkability_model": linkability_row["model"],
        "linkability_f1": linkability_row["f1_score"],
        "linkability_balanced_accuracy": linkability_row["balanced_accuracy"],
        "linkability_roc_auc": linkability_row["roc_auc"],
        "linkability_pr_auc": linkability_row["pr_auc"],
        "removed_features": removed_features,
    })

tradeoff_removal_df = pd.DataFrame(removal_results)
tradeoff_removal_df


,n_removed,n_kept,utility_model,utility_f1,utility_balanced_accuracy,utility_roc_auc,utility_pr_auc,linkability_model,linkability_f1,linkability_balanced_accuracy,linkability_roc_auc,linkability_pr_auc,removed_features
0,0,150,LogisticRegression,0.678106,0.831456,0.906530,0.710244,XGBoost,0.972864,0.97300,0.997051,0.997157,[]
1,10,140,LogisticRegression,0.676987,0.830770,0.905900,0.709645,XGBoost,0.972104,0.97225,0.996015,0.996268,"[lead_I_max, global_min_energy, global_mean_am..."
2,20,130,LogisticRegression,0.665033,0.823306,0.897739,0.688088,XGBoost,0.968648,0.96875,0.995232,0.995547,"[lead_I_max, global_min_energy, global_mean_am..."
3,30,120,LogisticRegression,0.653883,0.815811,0.891383,0.669873,XGBoost,0.963910,0.96400,0.994811,0.995060,"[lead_I_max, global_min_energy, global_mean_am..."
4,40,110,LogisticRegression,0.646042,0.810015,0.886265,0.659964,XGBoost,0.962833,0.96300,0.993679,0.994047,"[lead_I_max, global_min_energy, global_mean_am..."
5,50,100,LogisticRegression,0.635907,0.802330,0.877782,0.644980,XGBoost,0.959076,0.95925,0.993234,0.993642,"[lead_I_max, global_min_energy, global_mean_am..."


## Relative Change From Top-150 Baseline


In [7]:
baseline_row = tradeoff_removal_df[tradeoff_removal_df["n_removed"] == 0].iloc[0]
comparison_df = tradeoff_removal_df.copy()
comparison_df["utility_f1_drop"] = baseline_row["utility_f1"] - comparison_df["utility_f1"]
comparison_df["utility_ba_drop"] = baseline_row["utility_balanced_accuracy"] - comparison_df["utility_balanced_accuracy"]
comparison_df["linkability_roc_auc_drop"] = baseline_row["linkability_roc_auc"] - comparison_df["linkability_roc_auc"]
comparison_df[[
    "n_removed",
    "n_kept",
    "utility_f1",
    "utility_f1_drop",
    "utility_balanced_accuracy",
    "utility_ba_drop",
    "linkability_roc_auc",
    "linkability_roc_auc_drop",
]]


,n_removed,n_kept,utility_f1,utility_f1_drop,utility_balanced_accuracy,utility_ba_drop,linkability_roc_auc,linkability_roc_auc_drop
0,0,150,0.678106,0.000000,0.831456,0.000000,0.997051,0.000000
1,10,140,0.676987,0.001118,0.830770,0.000687,0.996015,0.001036
2,20,130,0.665033,0.013073,0.823306,0.008150,0.995232,0.001819
3,30,120,0.653883,0.024223,0.815811,0.015645,0.994811,0.002240
4,40,110,0.646042,0.032064,0.810015,0.021442,0.993679,0.003372
5,50,100,0.635907,0.042199,0.802330,0.029126,0.993234,0.003817


## Candidate Selection

Look for the smallest number of removed features that:
- causes only a small drop in utility;
- produces the largest possible reduction in linkability.


In [8]:
tradeoff_removal_df[["n_removed", "n_kept", "removed_features"]].head(len(REMOVAL_STEPS))


,n_removed,n_kept,removed_features
0,0,150,[]
1,10,140,"[lead_I_max, global_min_energy, global_mean_am..."
2,20,130,"[lead_I_max, global_min_energy, global_mean_am..."
3,30,120,"[lead_I_max, global_min_energy, global_mean_am..."
4,40,110,"[lead_I_max, global_min_energy, global_mean_am..."
5,50,100,"[lead_I_max, global_min_energy, global_mean_am..."
